# 02 Silver: page text extraction, OCR routing, data quality

| | |
|---|---|
| **Layer** | Silver |
| **Reads** | `bronze_doc_manifest` and the PDF files |
| **Writes** | `silver_doc_pages`, `dq_doc_pages` |
| **Re-runnable (idempotent)** | Yes. Documents already in Silver are skipped unless `FULL_REFRESH = True`. |

One Silver row per page. Native text is extracted with pdfplumber. If a page has almost no text (for example a scan), it is routed to Azure Document Intelligence OCR when an endpoint and key are configured. Every problem is logged to a data quality table instead of failing silently.

`doc_id` equals the file's SHA-256 from the manifest, so any page can be traced to its file, load time, and batch.

**OCR is optional.** Without `DOCINTEL_ENDPOINT` and `DOCINTEL_KEY`, near-empty pages are kept and flagged `EMPTY_PAGE`.

**Standards followed:** parameters at the top, no secrets in code, small documented functions, explicit schemas, a validation cell at the end, and a printed run summary.

## Parameters

In [ ]:
MANIFEST_TABLE   = "bronze_doc_manifest"
PAGES_TABLE      = "silver_doc_pages"
DQ_TABLE         = "dq_doc_pages"
LAKEHOUSE_ROOT   = "/lakehouse/default"

MIN_CHARS        = 50      # a page with fewer characters than this is treated as "no text layer"
CONF_THRESHOLD   = 0.80    # OCR average word confidence below this is flagged LOW_OCR_CONFIDENCE
OCR_DPI          = 150     # resolution used when rendering a page for OCR
X_TOLERANCE      = 2       # pdfplumber: smaller values split words that were merged ("GlennHarnett")
FULL_REFRESH     = False   # True = reprocess every document and overwrite both tables

# Testing aid: pretend these pages have no text layer, to exercise the OCR path.
# Example: {("my_file.pdf", 1)}
SIMULATE_SCANNED_PAGES = set()

## Setup
OCR credentials come from environment variables or a secret store. They are never typed into the notebook or committed.

In [ ]:
%pip install pdfplumber azure-ai-documentintelligence --quiet

In [ ]:
import io, os, uuid
from datetime import datetime
import pdfplumber
from pyspark.sql import functions as F
from pyspark.sql.window import Window

PAGES_SCHEMA = ("doc_id string, file_name string, page_number int, text string, char_count int, "
                "extraction_method string, ocr_confidence double, extracted_ts timestamp, batch_id string")
DQ_SCHEMA = ("doc_id string, file_name string, page_number int, dq_code string, detail string, "
             "checked_ts timestamp, batch_id string")

run_ts = datetime.utcnow()
batch_id = run_ts.strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:8]


def build_ocr_client():
    """Return an OCR client, or None when no credentials are configured (OCR is then skipped)."""
    endpoint, key = os.environ.get("DOCINTEL_ENDPOINT"), os.environ.get("DOCINTEL_KEY")
    if not (endpoint and key):
        return None
    from azure.ai.documentintelligence import DocumentIntelligenceClient
    from azure.core.credentials import AzureKeyCredential
    return DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))


ocr_client = build_ocr_client()
print("OCR available:", ocr_client is not None)

## Helpers

In [ ]:
def extract_native_text(page) -> str:
    """Extract the embedded text layer of one pdfplumber page ('' if none)."""
    return (page.extract_text(x_tolerance=X_TOLERANCE) or "").strip()


def ocr_page(client, page):
    """OCR one page. Returns (text, average word confidence or None)."""
    image = page.to_image(resolution=OCR_DPI).original            # PIL image
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    result = client.begin_analyze_document(
        "prebuilt-read", body=buffer.getvalue(), content_type="application/octet-stream").result()
    words = [w for p in (result.pages or []) for w in (p.words or [])]
    confidence = sum(w.confidence for w in words) / len(words) if words else None
    return (result.content or "").strip(), confidence


def process_document(doc_id: str, file_name: str, path: str):
    """Return (page_rows, dq_rows) for one PDF. Never raises for a bad page; it logs instead."""
    page_rows, dq_rows = [], []
    with pdfplumber.open(path) as pdf:
        for number, page in enumerate(pdf.pages, start=1):
            text = "" if (file_name, number) in SIMULATE_SCANNED_PAGES else extract_native_text(page)
            method, confidence = "native", None

            if len(text) < MIN_CHARS:                                # no usable text layer
                if ocr_client is not None:
                    try:
                        text, confidence = ocr_page(ocr_client, page)
                        method = "ocr"
                    except Exception as err:                         # log and continue with the next page
                        dq_rows.append((doc_id, file_name, number, "OCR_ERROR", str(err)[:300], run_ts, batch_id))
                        method = "none"
                else:
                    method = "none"

            if method == "ocr" and confidence is not None and confidence < CONF_THRESHOLD:
                dq_rows.append((doc_id, file_name, number, "LOW_OCR_CONFIDENCE",
                                f"confidence={confidence:.2f}", run_ts, batch_id))
            if len(text) < MIN_CHARS:
                dq_rows.append((doc_id, file_name, number, "EMPTY_PAGE",
                                f"chars={len(text)}, method={method}", run_ts, batch_id))

            page_rows.append((doc_id, file_name, number, text, len(text), method, confidence, run_ts, batch_id))
    return page_rows, dq_rows

## Select documents to process
One row per document (latest manifest row per hash). Already processed documents are skipped, which makes the notebook safe to rerun.

In [ ]:
manifest = spark.table(MANIFEST_TABLE)
latest = (manifest.withColumn("rn", F.row_number().over(
              Window.partitionBy("sha256").orderBy(F.col("load_ts").desc())))
          .filter("rn = 1").drop("rn"))

to_process = latest
if not FULL_REFRESH and spark.catalog.tableExists(PAGES_TABLE):
    done = spark.table(PAGES_TABLE).select(F.col("doc_id").alias("sha256")).distinct()
    to_process = latest.join(done, "sha256", "left_anti")

docs = to_process.collect()
print(f"documents to process: {len(docs)}")

## Extract, route, log

In [ ]:
all_pages, all_dq = [], []

# Duplicate files: the same content stored under more than one name.
dupes = (manifest.groupBy("sha256").agg(F.countDistinct("file_name").alias("names"))
                 .filter("names > 1").collect())
for d in dupes:
    all_dq.append((d["sha256"], None, None, "DUPLICATE_FILE", f"{d['names']} file names share this hash", run_ts, batch_id))

for doc in docs:
    pages, dq = process_document(doc["sha256"], doc["file_name"], os.path.join(LAKEHOUSE_ROOT, doc["file_path"]))
    all_pages.extend(pages)
    all_dq.extend(dq)
    print(f"{doc['file_name']}: {len(pages)} pages, {len(dq)} quality issues")

mode = "overwrite" if FULL_REFRESH else "append"
if all_pages:
    spark.createDataFrame(all_pages, PAGES_SCHEMA).write.format("delta").mode(mode).saveAsTable(PAGES_TABLE)
if all_dq:
    spark.createDataFrame(all_dq, DQ_SCHEMA).write.format("delta").mode(mode).saveAsTable(DQ_TABLE)

print(f"batch={batch_id} | pages written={len(all_pages)} | quality rows written={len(all_dq)}")

## Validation
- One row per page per document.
- `ocr_confidence` is NULL for natively extracted pages. That is correct: no OCR ran.
- Review quality issues by code.

In [ ]:
pages = spark.table(PAGES_TABLE)
print("pages:", pages.count(), "| documents:", pages.select("doc_id").distinct().count())
display(pages.groupBy("extraction_method").count())
display(spark.table(DQ_TABLE).groupBy("dq_code").count())

# Lineage: trace pages back to the Bronze file, load time, and batch.
display(pages.join(manifest.select(F.col("sha256").alias("doc_id"), "file_path", "load_ts"), "doc_id")
             .select("file_name", "page_number", "file_path", "load_ts").limit(10))